# Bölüm 17 — Speeding Up Transformers  
### Transformer'ları Hızlandırma Teknikleri

>  **Kaynak:** *Hands-On Machine Learning with Scikit-Learn and PyTorch* — Aurélien Géron  
>  **Açıklamalar:** Türkçe | **Terimler & Kod:** İngilizce  
>  Bu bölüm kitabın **yalnızca online** olarak sunulan ileri düzey bölümüdür.

---

## 📖 Bölümün Bağlamı

15. ve 16. bölümlerde çeşitli Transformer mimarileri inşa ettik: classifier, translator, chatbot, vision ve multimodal transformer'lar. Transformer'lar son derece çok yönlü ve güçlü olmakla birlikte **çok yavaş** olabilirler; özellikle uzun giriş dizilerini işlerken.

Neyse ki Transformer'ları hızlandırmak için pek çok teknik geliştirilmiştir. Bu bölümde şunları ele alacağız:

### 🗺️ Bölümün Haritası

```
1. INFERENCE HIZLANDIRMA (Faster Decoding)
   ├── Key/Value Caching      → Önceki adımlarda hesaplanan K/V'leri önbellekle
   └── Speculative Decoding   → Küçük taslak model + büyük doğrulama modeli

2. MULTI-HEAD ATTENTION HIZLANDIRMA
   ├── BigBird (Sparse)       → Her token yalnızca bir alt kümeye bakabilir → O(n)
   ├── Reformer (LSH)         → Benzer token'ları grupla, sadece grupla attention
   ├── Performer (FAVOR+)     → Kernel trick ile O(n·m) lineer attention
   ├── MQA                    → Tüm Q başlıkları tek K/V paylaşır
   ├── GQA                    → Q başlıkları gruplara ayrılır, her grup bir K/V
   └── FlashAttention         → IO-aware tiling, HBM trafiğini minimize et

3. BÜYÜK ÖLÇEKLI TRANSFORMER'LAR
   └── Mixture of Experts     → Sadece seçili expert'leri aktive et

4. EĞİTİM HIZLANDIRMA (Faster Training)
   ├── LoRA (PEFT)            → Düşük rankli güncelleme ile verimli fine-tuning
   └── Gradient Accumulation  → Küçük batch'leri biriktirerek büyük batch simülasyonu
```

>  **Not:** Transformer'ları küçültmenin bir diğer yolu da hassasiyeti azaltmak ve quantization uygulamaktır (Ek B'de ele alınmıştır).


---
##  BÖLÜM 1: Ortam Kurulumu (Setup)

Bu bölümdeki teknikler ciddi hesaplama gerektirmektedir.  
GPU (CUDA veya Apple MPS) olmadan denemek çok uzun sürebilir.


In [ ]:
import sys

# Bu bölüm Python 3.10 veya üzerini gerektirir.
# sys.version_info: (major, minor, micro, releaselevel, serial) tuple'ını döndürür.
# >= (3, 10) karşılaştırması: major=3 AND minor>=10 kontrolü yapar.
# Koşul sağlanmazsa AssertionError fırlatılır → notebook hemen hata verir.
assert sys.version_info >= (3, 10), (
    f"Python 3.10+ gerekli, mevcut: {sys.version_info.major}.{sys.version_info.minor}"
)
print(f"Python versiyonu: {sys.version_info.major}.{sys.version_info.minor} ✓")


In [ ]:
# sys.modules: Şu anda yüklü olan tüm Python modüllerinin sözlüğüdür.
# Bir modül adı bu sözlükte varsa → o ortamda çalışıyoruz demektir.
#
# IS_COLAB: Google Colab ortamında mıyız?
#   google.colab yalnızca Colab runtime'ında yüklenir.
# IS_KAGGLE: Kaggle Kernels ortamında mıyız?
#   kaggle_secrets yalnızca Kaggle'da kullanılabilir.
#
# Bu değişkenler sonraki hücrelerde:
#   - Paket kurulumu (pip install)
#   - GPU/TPU uyarıları
#   - API key yükleme gibi platform-spesifik işlemler için kullanılır.
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

print(f"Colab ortamı  : {IS_COLAB}")
print(f"Kaggle ortamı : {IS_KAGGLE}")


In [ ]:
from packaging.version import Version
import torch

# Bu bölüm için PyTorch 2.6.0 veya üzeri gereklidir.
# Nedeni: F.scaled_dot_product_attention(..., enable_gqa=True) argümanı
#         ve diğer gelişmiş attention API'leri bu versiyonda eklendi.
#
# packaging.version.Version sınıfı, versiyon string'lerini semantik
# karşılaştırmaya (major.minor.patch) uygun nesnelere çevirir.
# "2.6.0" > "2.5.1" gibi karşılaştırmalar düzgün çalışır.
assert Version(torch.__version__) >= Version("2.6.0"), (
    f"PyTorch 2.6.0+ gerekli, mevcut: {torch.__version__}"
)
print(f"PyTorch versiyonu: {torch.__version__} ✓")


In [3]:
# GPU olmadan bu bölümdeki transformer modelleri çok yavaş çalışır.
# Öncelik sırası: CUDA (NVIDIA GPU) > MPS (Apple Silicon) > CPU
#
# torch.cuda.is_available():
#   NVIDIA GPU ve CUDA driver'ı yüklü ise True döner.
#   Colab/Kaggle'da GPU runtime seçilmişse True.
#
# torch.backends.mps.is_available():
#   Apple M1/M2/M3 çipli Mac'lerde Metal Performance Shaders kullanımı.
#   PyTorch 1.12+ ile desteklenir.
#
# "cpu": GPU yoksa son çare — bazı hücreler dakikalarca sürebilir.
if torch.cuda.is_available():
    device = "cuda"    # NVIDIA GPU — en hızlı seçenek
elif torch.backends.mps.is_available():
    device = "mps"     # Apple Silicon GPU
else:
    device = "cpu"     # Son çare — yavaş

print(f"Seçilen device: {device}")
device


Seçilen device: mps


'mps'

In [4]:
# Eğer GPU bulunamadıysa kullanıcıyı uyar ve nasıl GPU
# aktive edeceğini platform bazlı açıkla.
if device == "cpu":
    print("⚠️  Neural nets can be very slow without a hardware accelerator.")
    if IS_COLAB:
        # Colab'da: Runtime > Change runtime type > Hardware accelerator > GPU
        print("👉 Go to Runtime > Change runtime and select a GPU hardware accelerator.")
    if IS_KAGGLE:
        # Kaggle'da: Settings sekmesinden Accelerator açılabilir
        print("👉 Go to Settings > Accelerator and select GPU.")


In [5]:
import matplotlib.pyplot as plt

# Tüm grafiklerde tutarlı ve okunabilir yazı boyutları için
# global Matplotlib ayarları yapılır.
# plt.rc(group, **kwargs): Belirtilen grupta varsayılan değerleri değiştirir.
# Bu ayarlar notebook boyunca geçerli olacak; her plt çağrısında tekrar belirtmek gerekmez.
plt.rc('font', size=14)                      # Genel yazı boyutu
plt.rc('axes', labelsize=14, titlesize=14)   # Eksen etiketleri ve başlık
plt.rc('legend', fontsize=14)                # Lejant yazı boyutu
plt.rc('xtick', labelsize=10)                # X ekseni tick etiketleri
plt.rc('ytick', labelsize=10)                # Y ekseni tick etiketleri
print("Matplotlib ayarları yapıldı ✓")


Matplotlib ayarları yapıldı ✓


In [2]:
!pip uninstall -y scikit-learn
!pip install scikit-learn==1.3.2  # NumPy 1.26 ile en uyumlu sürümlerden biri

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 29.9 kB/s  0:04:17m0:00:0100:11


---
##  BÖLÜM 2: Inference Zamanında Daha Hızlı Decoding

### Autoregressive Generation'ın Yavaşlık Problemi

GPT, OPT, Llama gibi **causal language model**'lar metin üretirken  
**autoregressive** (otobağlanımlı) bir yaklaşım kullanır:  
her adımda yalnızca **bir token** üretilir ve bu token bir sonraki  
adımda giriş olarak kullanılır.

**500 token üretmek = 500 kez forward pass çalıştırmak**

Her forward pass sırasında:
- Tüm geçmiş token'ların **Key (K)** ve **Value (V)** tensörleri  
  tüm Transformer katmanlarında hesaplanır.
- Bu hesaplar bir önceki adımla **tamamen aynıdır** — saf tekrar!

```
Adım 1: [t₁]           → K₁,V₁ hesapla → t₂ üret
Adım 2: [t₁, t₂]       → K₁,V₁, K₂,V₂ hesapla → t₃ üret  (K₁,V₁ tekrar!)
Adım 3: [t₁, t₂, t₃]   → K₁,V₁,K₂,V₂,K₃,V₃ hesapla → t₄ (K₁,V₁,K₂,V₂ tekrar!)
...
Adım n: tüm geçmiş → yeniden hesap → O(n²) toplam maliyet!
```


### 2.1 Key/Value (KV) Caching

#### Temel Fikir

Her adımda **yalnızca yeni token'ın** K ve V değerlerini hesapla.  
Önceki tüm token'ların K ve V değerleri bir **önbellekte (cache)** tutulur  
ve yalnızca yeni token'ın K,V'si bu cache'e eklenir.

#### Cache Olmadan vs. Cache İle

| | Cache yok | Cache var |
|---|:---|:---|
| Adım 1 | K₁,V₁ hesapla | K₁,V₁ hesapla → cache'e ekle |
| Adım 2 | **K₁,V₁, K₂,V₂ hesapla** | **sadece K₂,V₂** hesapla → cache: [K₁,V₁, K₂,V₂] |
| Adım n | K₁…Kₙ,V₁…Vₙ hesapla | **sadece Kₙ,Vₙ** hesapla → cache'e ekle |

**Tasarruf:** n token üretmek için O(n²) → O(n) hesaplama!

#### KV Cache'in Maliyeti: Bellek

Cache, GPU VRAM'inde yer kaplar.  
Her katman için cache boyutu: `2 × seq_len × d_model × n_layers × dtype_bytes`

Örnek — Llama-3 8B, float16, 8K context:
```
2 × 8192 × 4096 × 32 × 2 byte ≈ 4 GB sadece KV cache için!
```
Bu yüzden uzun context'lerde KV cache bellek sorununa yol açabilir.  
Çözümler: Quantized KV cache, sliding window attention, vb.

#### `use_cache` Parametresi

Hugging Face `generate()` metodunda `use_cache=True` (varsayılan) ile  
KV caching otomatik olarak aktif edilir.  
`use_cache=False` ile devre dışı bırakarak fark ölçülebilir.


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Model Seçimi ──
# facebook/opt-125m: Meta'nın açık kaynak OPT ailesinden küçük model.
# 125 milyon parametre — demo için ideal boyut.
# "opt" = Open Pre-trained Transformer
# "125m" = 125 milyon parametre
model_id = "facebook/opt-125m"

# AutoModelForCausalLM: Causal (tek yönlü) dil modeli.
# "Causal" = her token yalnızca kendinden önceki token'lara bakabilir.
# device_map="auto": Uygun device'ı (cuda/mps/cpu) otomatik seç.
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

# AutoTokenizer: Model ile eşleşen tokenizer'ı yükle.
# Tokenizer metin → token ID dizisi → metin dönüşümünü yapar.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# ── Test Prompt'u ──
prompt = "Once upon a time there lived"

# tokenizer(prompt, return_tensors="pt"):
#   - Metni token ID'lerine çevirir
#   - return_tensors="pt": PyTorch tensor olarak döndür
#   - .to(model.device): Modelin bulunduğu device'a taşı
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# ── KV Cache Karşılaştırması ──
for use_cache in (True, False):
    print(f"{use_cache=}")  # f-string debug notasyonu: "use_cache=True" yazar
    # %time: IPython magic komutu — ifadenin çalışma süresini ölçer ve gösterir
    # max_new_tokens=500: Prompt'tan sonra 500 yeni token üret
    # do_sample=False: Greedy decoding — her adımda en yüksek olasılıklı token
    #                  (deterministik, her çalışmada aynı sonuç)
    # use_cache=True : KV önbelleği kullan → her adımda yalnızca yeni K/V hesaplanır
    # use_cache=False: KV önbelleği kullanma → her adımda TÜM geçmiş yeniden hesaplanır
    %time model.generate(**inputs, max_new_tokens=500, do_sample=False, use_cache=use_cache)

# Beklenen sonuç (GPU'da):
# use_cache=True  → ~2-5 saniye
# use_cache=False → ~10-30 saniye (veya daha fazla, dizi uzadıkça katlanarak yavaşlar)
# CPU'da her ikisi de çok daha yavaş olur.


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /facebook/opt-125m/resolve/main/config.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fcab0743040>: Failed to establish a new connection: [Errno 8] nodename nor servname provided, or not known'))"), '(Request ID: 3b723a87-6ba7-4074-9fd1-6ced6ff2a3fe)')' thrown while requesting HEAD https://huggingface.co/facebook/opt-125m/resolve/main/config.json
Retrying in 1s [Retry 1/5].


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

---
### 2.2 Speculative Decoding — Taslak + Doğrulama

#### Motivasyon

KV caching, önceki hesapları tekrar etmez ama her adımda hâlâ  
yalnızca **bir token** üretilir ve büyük modelin tüm ağırlıkları çalıştırılır.

**Büyük model (target):** 7B, 70B parametre — kaliteli ama yavaş  
**Küçük model (draft):** 125M, 350M parametre — hızlı ama düşük kalite  

**Speculative Decoding** bu iki modelin güçlü yanlarını birleştirir.

#### Algoritma — Adım Adım

```
1. [DRAFT PHASE] Draft model γ adet token üretir (örn. γ=5)
   Draft: t₁, t₂, t₃, t₄, t₅  ← hızlı, paralel üretim mümkün

2. [VERIFY PHASE] Target model bu 5 token'ı TEK bir forward pass'te doğrular
   Target: p(t₁|context), p(t₂|t₁), p(t₃|t₁,t₂), ... hepsini aynı anda hesapla

3. [ACCEPT/REJECT] Token kabul/red kararı
   - p_target(tᵢ) / p_draft(tᵢ) > random(0,1) → KABUL
   - İlk red noktasından sonraki tüm token'lar atılır
   - Target model bu noktadan devam eder

4. Döngü: yeni draft tokens → verify → accept/reject...
```

#### Neden Hızlı?

Büyük model normalde 5 token için 5 forward pass yapar.  
Speculative decoding ile: 1 forward pass'te 5 token'ı doğrula!

GPU büyük matris işlemlerinde en verimlidir — küçük batch'ler GPU'yu boş bırakır.  
5 token'lık batch → daha iyi GPU kullanımı → hızlanma!

#### Neden Güvenli (Kalite Koruma)?

Matematiksel kanıt: Kabul/red mekanizması, çıktı dağılımının  
**target modelin dağılımına eşit** olduğunu garanti eder.  
Başka bir deyişle: speculative decoding yalnızca target modeli çalıştırmakla  
**tamamen aynı kalitede** çıktı üretir — sadece daha hızlı!

#### Gereksinimler

- Draft ve target model **aynı tokenizer** kullanmalı (aynı kelime dağarcığı)
- Genellikle aynı modelin farklı boyutları kullanılır:
  - OPT-125M (draft) + OPT-350M (target)
  - Llama-3-1B (draft) + Llama-3-8B (target)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import set_seed

# Tekrarlanabilirlik için global rastgelelik tohumunu sabitle.
# set_seed(42): PyTorch, NumPy ve Python random modüllerini aynı anda ayarlar.
# Bu sayede her çalıştırmada aynı token üretilir (do_sample=True olmasına rağmen).
set_seed(42)

# ── Target Model (Büyük, Kaliteli) ──
# facebook/opt-350m: 350 milyon parametre — "asıl" model.
# Bu modelin ürettiği kaliteyi korumak istiyoruz.
target_model = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-350m",
    device_map="auto"   # GPU varsa GPU'ya yükle
)

# ── Draft Model (Küçük, Hızlı) ──
# facebook/opt-125m: 125 milyon parametre — "taslak" model.
# Target'tan ~3x daha az parametre → ~3x daha hızlı forward pass.
# OPT ailesi aynı tokenizer'ı paylaşır → speculative decoding için ideal.
draft_model = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-125m",
    device_map="auto"
)

# Target modelin tokenizer'ını kullan (her ikisi de aynı)
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m")

# ── Speculative Decoding ile Üretim ──
prompt = "Once upon a time there lived"
inputs = tokenizer(prompt, return_tensors="pt").to(target_model.device)

outputs = target_model.generate(
    **inputs,
    max_new_tokens=100,          # Toplam 100 yeni token üret
    do_sample=True,              # Örnekleme tabanlı üretim (deterministik değil)
    temperature=1,               # temperature=1: Orijinal olasılık dağılımı
                                 # < 1: Daha tutarlı/tahmin edilebilir
                                 # > 1: Daha yaratıcı/rastgele

    assistant_model=draft_model  # ← Speculative Decoding'i aktive eden parametre!
                                 # Hugging Face bu argümanı görünce otomatik olarak:
                                 # 1. draft_model ile γ token üretir (varsayılan γ=5)
                                 # 2. target_model ile tek forward pass'te doğrular
                                 # 3. Kabul/red uygular
                                 # 4. Döngüyü tekrarlar
)

# Üretilen token ID'lerini metne çevir
# outputs[0]: Batch'ten ilk (tek) örneği al
# skip_special_tokens=True: <s>, </s>, <pad> gibi özel token'ları atla
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

# Beklenen: Tutarlı, devam eden bir hikaye cümlesi
# Kalite: Yalnızca opt-350m çalıştırılmış gibi (matematiksel garanti)
# Hız: Draft model sayesinde genellikle 2-4x daha hızlı


---
## BÖLÜM 3: Multi-Head Attention'ı Hızlandırma

### Neden MHA Darboğazdır?

Transformer'ın en pahalı bileşeni **Multi-Head Attention (MHA)**'dır.

#### Standart MHA Karmaşıklığı

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

- $QK^\top$ matrisi: $(L_q \times L_k)$ — **O(n²) bellek ve zaman**
- $n = L_q = L_k$ (sequence uzunluğu) olduğunda:

| Sequence uzunluğu | Attention matrisi |
|:-:|:-:|
| 512 token | 512² = **262 K** eleman |
| 2,048 token | 2048² = **4.2 M** eleman |
| 8,192 token | 8192² = **67 M** eleman |
| 128,000 token | ~**16 Milyar** eleman — imkânsız! |

#### Çözüm Yolları

| Yöntem | Fikir | Karmaşıklık |
|--------|-------|:-----------:|
| BigBird | Sparse attention — sadece bazı çiftler | O(n) |
| Reformer | LSH ile benzer Q/K'ları grupla | O(n log n) |
| Performer | Kernel trick ile matris bölünmesi | O(n·m) |
| MQA/GQA | K/V başlıklarını paylaş | O(n²/h) |
| FlashAttention | Aynı hesap, farklı bellek erişim düzeni | O(n²), ama IO↓ |


---
### 3.1 BigBird — Sparse Attention

#### Tam (Dense) Attention Problemi

Standart MHA'da her token **diğer tüm token'lara** attention yapar:  
`Token_i → Token_1, Token_2, ..., Token_n` — O(n²)

#### BigBird'ün Yaklaşımı: 3 Tür Sparse Bağlantı

BigBird, her token'ı yalnızca **üç kümeyle** bağlantılandırır:

```
1. LOCAL (Yerel pencere):
   Her token, yakın komşularına bakabilir (sliding window, genişlik w).
   Dil modellerinde yakın kelimeler güçlü ilişkilidir.
   → Her token w token'a bakabilir

2. GLOBAL TOKEN'LAR:
   Bazı özel token'lar (örn. [CLS]) tüm dizi ile etkileşime girer.
   Bu token'lar "hub" görevi görür — uzak bağımlılıkları taşır.
   → g global token, herkesle etkileşir

3. RANDOM (Rastgele):
   Her token r adet rastgele seçilmiş token'a bakabilir.
   Uzak bağımlılıklar için stochastic köprü sağlar.
   → Her token r rastgele token'a bakabilir
```

**Her token için toplam attention sayısı:** w + g + r (sabit, n'den bağımsız!)  
**Karmaşıklık:** O(n · (w + g + r)) ≈ **O(n)**

#### Teorik Yeterlilik

BigBird makalesi, bu sparse yapının **teorik olarak tam attention ile eşdeğer**  
olduğunu kanıtlar — yani Turing Complete. Büyük bağımlılıklar  
global token'lar ve rastgele bağlantılar üzerinden iletilir.

#### BigBird Modeli

`google/bigbird-roberta-base` = BERT/RoBERTa mimarisi + BigBird attention  
Context uzunluğu: **4,096 token** (BERT'in 512'sinin 8 katı)


In [ ]:
from transformers import pipeline

# google/bigbird-roberta-base:
#   - "bigbird": BigBird sparse attention mekanizması kullanır
#   - "roberta": RoBERTa (BERT geliştirilmiş versiyonu) mimarisi
#   - "base": Temel boyut (large'ın aksine)
# BigBird'ün asıl gücü uzun belgelerde (hukuki metin, bilimsel makale vb.) ortaya çıkar.
# Kısa cümleler için BERT ile benzer performans gösterir.
model_id = "google/bigbird-roberta-base"

# pipeline: Belirli bir NLP görevini tek satırda çalıştırmanın kolay yolu.
# task="fill-mask": Metindeki [MASK] token'ını en olası kelimeyle doldur.
# Masked Language Modeling (MLM) — BERT ailesinin eğitim görevi.
pipeline = pipeline(task="fill-mask", model=model_id)

# Test cümlesi: [MASK] token'ını tahmin ettir
# "She was feeling unwell so she took some [MASK] medicine."
# Beklenen tahminler: "cold", "flu", "headache", "pain" relief medicine vb.
result = pipeline("She was feeling unwell so she took some [MASK] medicine.")

print("BigBird Fill-Mask Tahminleri:")
for pred in result:
    print(f"  {pred['token_str']:15s} → {pred['score']:.2%}")

# BigBird burada standart BERT gibi çalışır.
# Farkı yalnızca uzun dizilerde (>512 token) belirginleşir:
# BERT → hata/kesme, BigBird → 4096 token'a kadar sorunsuz çalışır.


> ⚠️ **Not:** Yukarıdaki uyarı mesajları BigBird'ün Hugging Face entegrasyonundan kaynaklanmaktadır, güvenle görmezden gelebilirsiniz.

---
### 3.2 Approximate Attention — Reformer ve LSH

#### Sparse → Approximate: Fark Nedir?

- **Sparse attention** (BigBird): Kesin kurallara göre bazı çiftleri atlar
- **Approximate attention** (Reformer, Performer): Tam hesap yerine **yaklaşık** sonuç

#### Reformer'ın Temel Gözlemi

Standart softmax attention'da matrisin büyük çoğunluğu **sıfıra yakındır**:

$$A_{ij} = 	ext{softmax}\!\left(rac{Q_i K_j^\top}{\sqrt{d}}\right) pprox 0 \quad 	ext{(çoğu } i,j 	ext{ çifti için)}$$

Neden? Softmax'ta yalnızca **Q'ya en yakın K'lar** yüksek ağırlık alır.  
Uzak (düşük dot product) çiftlerin katkısı ihmal edilebilir.

**Reformer'ın fikri:** Q ve K'ları önceden **gruplara (bucket)** ayır.  
Yalnızca aynı gruptaki Q ve K çiftleri için attention hesapla.  
Benzer Q ve K'lar aynı gruba düşüyorsa → ihmal edilen çiftler gerçekten önemsizdi!

Bu gruplaşma için **Locality-Sensitive Hashing (LSH)** kullanılır.

#### Locality-Sensitive Hashing (LSH) — Kavramsal Bakış

LSH, benzer vektörlerin **yüksek olasılıkla aynı hash (bucket) değerine**  
düşeceği bir hash fonksiyonu ailesidir:

```
Benzer vektörler  → Büyük olasılıkla AYNI bucket
Farklı vektörler  → Büyük olasılıkla FARKLI bucket
```

**Angular LSH** (Reformer'ın kullandığı versiyon):
- "Benzer" = yüksek dot product = küçük açı
- Rastgele hiperplanlarla uzayı bölme

#### Angular LSH Algoritması

```
1. Rastgele projeksiyon matrisi R oluştur: (d, k//2)
2. Vektörleri birim küreye normalize et: v̂ = v / ||v||
3. Projeksiyon hesapla: P = v̂ @ R  → (n, k//2)
4. Negatif projeksiyonu ekle: concat([P, -P]) → (n, k)
5. argmax ile bucket ID ata: bucket = argmax(concat([P, -P]))
```

**Neden `-P` ekliyoruz?**  
Karşıt yönlü vektörlerin (`v` ve `-v`) farklı bucket'a düşmesini sağlar.  
`[P, -P]` ile k bucket mümkündür ve karşıt vektörler birbirinden uzaklaşır.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def angular_lsh(vectors, k):
    """
    Angular Locality-Sensitive Hashing.

    Dot product'a göre birbirine yakın vektörleri yüksek olasılıkla
    aynı bucket'a atar. Reformer'da Q ve K'yı gruplamak için kullanılır:
    aynı bucket'taki Q ve K çiftleri attention yapar, diğerleri atlanır.

    Bu yaklaşım sayesinde O(n²) → O(n log n) karmaşıklığa düşülür.

    Parametreler:
    - vectors: (n, d) — hash'lenecek vektörler (sorgular Q veya anahtarlar K)
    - k:       Bucket sayısı — k adet farklı hash değeri (0 ... k-1)
               Büyük k → daha az çakışma, daha az yanlış eşleşme

    Döndürür:
    - (n,) int tensor — her vektörün atandığı bucket ID'si
    """
    # Adım 1: Rastgele projeksiyon matrisi
    # R: (d, k//2) — Gaussian dağılımdan örneklenen rastgele hiperplanlar
    # Her sütun, uzayı ikiye bölen bir rastgele hiperplanı temsil eder.
    # device=vectors.device: R'yi vektörlerle aynı device'ta oluştur (GPU uyumluluğu)
    R = torch.randn(vectors.size(-1), k // 2, device=vectors.device)

    # Adım 2: L2 Normalizasyon — vektörleri birim küreye yansıt
    # Dot product yerine angular (açısal) distance kullanmak için gerekli.
    # F.normalize(v, p=2.0, dim=1): Her satır vektörünün L2 normu 1 olacak şekilde ölçekle
    # Normalizasyon sonrası: cosine similarity = dot product
    normalized_vectors = F.normalize(vectors, p=2.0, dim=1)  # (n, d)

    # Adım 3: Projeksiyon
    # V_proj: (n, d) @ (d, k//2) = (n, k//2)
    # Her vektör k//2 boyutlu bir "imza"ya dönüşür.
    V_proj = normalized_vectors @ R  # (n, k//2)

    # Adım 4: Negatif projeksiyon ekle
    # V_concat: (n, k//2) ile (n, k//2) yatay olarak birleştirilir → (n, k)
    # -V_proj: Her projeksiyon değerinin negatifini ekle
    # Bu sayede:
    #   - k bucket yerine 2*(k//2) = k bucket elde edilir
    #   - Zıt yönlü vektörler (v ve -v) kesinlikle farklı bucket'a düşer
    #   - Her bucket, benzer yönlü vektörlerin toplandığı bir "küme" olur
    V_concat = torch.cat([V_proj, -V_proj], dim=1)  # (n, k)

    # Adım 5: Bucket ataması
    # argmax: Hangi bucket değeri en büyükse, vektör o bucket'a atanır.
    # Benzer (yüksek dot product) vektörler → benzer projeksiyon değerleri
    # → aynı argmax → aynı bucket!
    return torch.argmax(V_concat, dim=1)  # (n,) — her vektör için bucket ID


In [ ]:
# Angular LSH'i test et
torch.manual_seed(42)

# 16 adet 512-boyutlu rastgele vektör
vectors = torch.rand(16, 512)  # (16, 512)

# k=4: 4 farklı bucket (0, 1, 2, 3)
bucket_ids = angular_lsh(vectors, k=4)

print(f"Bucket ID'leri:")
print(bucket_ids)
print()
print(f"Her bucket'taki vektör sayısı:")
counts = torch.bincount(bucket_ids, minlength=4)
for bucket_id, count in enumerate(counts):
    print(f"  Bucket {bucket_id}: {count.item()} vektör")

# Reformer'da nasıl kullanılır:
# 1. Q ve K için bucket ID'leri hesapla
# 2. Q ve K'yı bucket ID'sine göre sırala
# 3. Sadece aynı bucket'taki Q-K çiftleri için attention hesapla
# 4. Bu işlem O(n log n) karmaşıklıkta yapılabilir (sıralama O(n log n))


---
### 3.3 Performer — FAVOR+ ile Lineer Attention

#### Sparse Attention'dan Farkı

Reformer ve BigBird bazı Q-K çiftlerini **atlar** (sparse).  
Performer ise **tüm çiftleri dikkate alır** ama hesabı **lineer** karmaşıklığa indirir!

Bu mucizevi gelişme **kernel trick** sayesinde mümkündür.

#### Kernel Trick Nedir?

Softmax attention'da:
$$A_{ij} = rac{\exp(Q_i K_j^\top / \sqrt{d})}{\sum_k \exp(Q_i K_k^\top / \sqrt{d})}$$

Eğer şunu yazabilseydik:
$$\exp(Q_i K_j^\top) = \phi(Q_i) \cdot \phi(K_j)^\top$$

o zaman:
$$\text{Attention}(Q,K,V) = \frac{\phi(Q)(\phi(K)^\top V)}{\phi(Q)\Sigma\phi(K)}$$

**Hesaplama sırası** kritik:
- Önce $\phi(K)^\top V$: $(m \times L_k) @ (L_k \times d) = (m \times d)$ — **O(L_k)**
- Sonra $\phi(Q) @ (m \times d)$: $(L_q \times m) @ (m \times d)$ — **O(L_q)**
- **Büyük $(L_q \times L_k)$ matrisi hiç oluşturulmaz!**
- Toplam karmaşıklık: **O((L_q + L_k) · m · d)** — lineer!

#### FAVOR+ Matematiksel Temeli

Gaussian örnekleme ile **w** vektörü için şu eşitlik sağlanır:

$$E_{\mathbf{w} \sim \mathcal{N}(0,I)}[\exp(\mathbf{w} \cdot \mathbf{x})] = \exp\!\left(\frac{1}{2}\|\mathbf{x}\|^2\right)$$

**Sezgisel anlamı:** Rastgele bir Gaussian vektörü ile iç çarpımın beklentisi,  
orijinal vektörün karenorm'unun yarısının üssüne eşittir.  
Bu gerçeği büyük m değerleri için ampirik olarak doğrulayabiliriz.


In [ ]:
# Gaussian dağılım özelliğini doğrula:
# E[exp(w·x)] = exp(||x||² / 2)   (w ~ N(0, I) için)
torch.manual_seed(42)

d = 64     # Vektör boyutu
m = 1024   # Örnekleme sayısı — büyük m → Büyük Sayılar Kanunu → iyi yaklaşım

# W: (d, m) — Gaussian dağılımdan örneklenen m adet d-boyutlu rastgele vektör
# Her sütun bağımsız bir w ~ N(0, I) örneklemesidir
W = torch.randn(d, m)

# X: 5 adet test vektörü
# / d**0.5 ile ölçekle: Çok büyük değerleri önle (exp taşması riski)
X = torch.randn(5, d) / d ** 0.5  # (5, d)

# R = exp(X @ W): Her vektör x için m adet w ile exp(w·x) hesapla
# X @ W: (5, d) @ (d, m) = (5, m) — her (x, w) çifti için x·w skaler değer
# torch.exp: Element-wise üs alma
R = torch.exp(X @ W)  # (5, m)

# Ampirik ortalama: m örnekleme üzerinden E[exp(w·x)] ≈ R.mean(axis=-1)
# axis=-1: Son boyut (m örnekler) üzerinden ortalama → (5,) vektörü
print("Ampirik E[exp(w·x)] (m=1024 örnekle):")
print(R.mean(axis=-1))

# Teorik değer: exp(||x||² / 2)
# X.norm(dim=-1): Her satırın L2 normu → (5,)
# **2: Karesini al
# 0.5 *: 1/2 ile çarp
# torch.exp: Üs al
print()
print("Teorik exp(||x||² / 2):")
print(torch.exp(0.5 * (X.norm(dim=-1)**2)))

# İki değer birbirine çok yakın olmalı!
# Fark: Monte Carlo hatası — m arttıkça küçülür (1/√m oranında)
# m=1024 için yaklaşık %3-5 hata beklenir


**Sonuç:** Değerler birbirine çok yakın!

Şimdi FAVOR+ feature map fonksiyonu $\phi$'yi implemente edelim:

$$\phi(\mathbf{x}) = \frac{\exp\!\left(\mathbf{x}\mathbf{W} - \max - \frac{1}{2}\|\mathbf{x}\|^2\right)}{\sqrt{m}}$$

**Neden bu $\phi$ işe yarıyor?**

Beklenti hesabından hareketle:
$$\phi(Q_i) \cdot \phi(K_j)^T \approx \exp(Q_i K_j^T)$$

Yani iki feature map'in dot product'ı, orijinal attention skorunu yaklaşık olarak verir!

**Formülde ne işe yarıyor?**

- `x @ W`: Vektörü m boyutlu feature uzayına projeksiyon yap
- `- max_vals`: Numerik stabilite — exp taşmasını önle (log-sum-exp trick)
- `- ||x||² / 2`: Yukarıdaki beklenti denklemini sağlamak için gereken normalizasyon
- `/ √m`: Beklenti değerini normalize et (m örneklemin etkisini kaldır)


In [ ]:
def phi(X, W, dim_subtract_max=(-2, -1)):
    """
    FAVOR+ (Fast Attention Via Positive Orthogonal Random features) feature map.

    Bu fonksiyon exp(Q·Kᵀ)'yi kernel trick ile yaklaşık hesaplar:
        φ(Q) @ φ(K)ᵀ ≈ exp(Q·Kᵀ)

    Bu sayede attention O(n²) yerine O(n·m) karmaşıklıkta hesaplanabilir.

    Parametreler:
    - X:                  (..., d) — feature map uygulanacak tensör (Q veya K)
    - W:                  (n_heads, d_head, m) — rastgele projeksiyon matrisi
    - dim_subtract_max:   Numerik stabilite için max'ın hangi boyutlar üzerinden
                          alınacağı:
                          * Q için: (-1,) — her query'nin kendi max'ı kullanılır
                            (Her Q kendi içinde normalize edilir)
                          * K için: (-2, -1) — tüm K'ların global max'ı kullanılır
                            (Aynı normalizasyon referansı → tutarlı kernel değerleri)

    Döndürür:
    - (..., m) — feature map çıktısı (her giriş vektörü m-boyutlu vektöre dönüşür)

    Formül: φ(X) = exp(X @ W - max_val - ||X||²/2) / √m
    """
    # Adım 1: Karenorm hesapla — ||X||² / 2
    # X.square(): Element-wise kare al
    # .sum(dim=-1): Son boyut üzerinden topla → her vektörün ||x||²
    # keepdim=True: Boyutu koru → broadcast için gerekli
    squared_norms = X.square().sum(dim=-1, keepdim=True)  # (..., 1)

    # Adım 2: Projeksiyon hesapla — X @ W
    # (..., d) @ (n_heads, d_head, m) → matmul broadcast kurallarıyla (..., m)
    X_proj = X @ W  # (..., m)

    # Adım 3: Max değeri çıkar (numerik stabilite)
    # Log-sum-exp trick: exp(a) - exp(b) = exp(a - max) - exp(b - max)
    # Max çıkartmak matematiksel değeri değiştirmez ama taşmayı önler
    # amax: Belirtilen boyutlar üzerinden maksimum değeri al
    max_vals = X_proj.amax(dim=dim_subtract_max, keepdim=True)  # (..., 1)

    # Adım 4: Feature map hesapla
    # exp(X @ W - max - ||X||²/2) / √m
    # W.size(-1): m değeri (projeksiyon sayısı)
    # / m**0.5: 1/√m normalizasyonu
    return torch.exp(X_proj - max_vals - squared_norms / 2) / W.size(-1) ** 0.5


### 3.4 φ(Q)φ(K)ᵀ ≈ softmax(QKᵀ) — Teorik Garantiyi Doğrula

$E[\phi(\mathbf{Q})\phi(\mathbf{K})^\top] = \exp(\mathbf{Q}\mathbf{K}^\top)$ olduğu ispat edilebilir.

Ampirik olarak kontrol edelim: `φ(Q) @ φ(K)ᵀ` ile gerçek `softmax(QKᵀ/√d)` ne kadar yakın?


In [ ]:
torch.manual_seed(42)

# ── Boyutlar ──
batch_size = 32    # Batch büyüklüğü
d_model = 512      # Toplam embedding boyutu
n_heads = 8        # MHA başlık sayısı
Lq = 200           # Query sequence uzunluğu
Lk = 100           # Key sequence uzunluğu
m = 256            # Random feature sayısı (Performer'da "özellik sayısı")
                   # Büyük m → daha iyi yaklaşım, daha fazla bellek/hesaplama
d_head = d_model // n_heads  # 512 / 8 = 64 — her başlığın boyutu

# ── Parametreler ──
# W: Her başlık için (d_head, m) projeksiyon matrisi
W = torch.randn(n_heads, d_head, m)  # (8, 64, 256)

# Rastgele Q ve K
Q = torch.randn(batch_size, n_heads, Lq, d_head)  # (32, 8, 200, 64)
K = torch.randn(batch_size, n_heads, Lk, d_head)  # (32, 8, 100, 64)

# Standart attention ölçek faktörü
scale = 1 / d_head ** 0.5  # 1/√64 = 0.125

# ── FAVOR+ Approximate Attention ──
# Kritik: scale**0.5 ile ölçekle — φ(√scale·Q) @ φ(√scale·K)ᵀ ≈ exp(scale·Q·Kᵀ)
# Neden scale**0.5? φ'nin içindeki exp quadratik terim nedeniyle:
#   φ(√scale·X) → ||√scale·X||² = scale·||X||² → exp(-scale·||X||²/2)
# dim_subtract_max farkı:
#   Q için (-1): Her query kendi içinde normalize edilir (local max)
#   K için (-2,-1): Tüm K'lar aynı global max kullanır (tutarlı normalizasyon)
Qp = phi(Q * scale ** 0.5, W, dim_subtract_max=-1)   # (32, 8, 200, 256)
Kp = phi(K * scale ** 0.5, W)                         # (32, 8, 100, 256)

# ── Lineer Attention Hesabı ──
# A ≈ exp(Q·Kᵀ·scale): (32,8,200,256) @ (32,8,256,100) = (32,8,200,100)
A = Qp @ Kp.transpose(-2, -1)

# Normalize: Softmax paydasını taklit et
# Her query için satır toplamına böl
D = A.sum(dim=-1, keepdim=True)  # (32, 8, 200, 1) — payda
result = A / (D + 1e-6)          # +1e-6: Sıfıra bölmeyi önle

# ── Gerçek Softmax Attention (referans) ──
# Bu standart hesap O(Lq × Lk) bellek kullanır
expected = torch.softmax(Q @ K.transpose(-2, -1) * scale, dim=-1)
# (32,8,200,64) @ (32,8,64,100) * 0.125 → softmax → (32,8,200,100)

# ── RMSE Karşılaştırması ──
# F.mse_loss: Ortalama karesel hata
# ** 0.5: Karekök → RMSE (Root Mean Square Error)
rmse = F.mse_loss(result, expected) ** 0.5
print(f"FAVOR+ RMSE (W rastgele, m=256): {rmse.item():.6f}")
# Tipik değer: 0.01 - 0.05 arası — pratikte kabul edilebilir
# m artırılırsa RMSE azalır (daha iyi yaklaşım)


**İyi bir yaklaşım!**

W matrisini **QR dekompozisyonuyla ortogonalize** ederek bunu daha da iyileştirebiliriz.

#### Neden Orthogonalizasyon?

Rastgele W sütunları birbirleriyle korelasyonlu olabilir.  
Korelasyon = bilgi tekrarı = projeksiyon verimsizliği.

**Çözüm:** W sütunlarını birbirine dik yaparsak maksimum bilgi çeşitliliği elde ederiz.

Ancak d-boyutlu uzayda en fazla d adet ortogonal vektör vardır.  
m > d ise W'yi d'lik chunk'lara böl, her chunk'u bağımsız ortogonalize et.

**QR Dekompozisyonu:** $A = QR$ (buradaki Q, dikkat matrisinden farklı!)  
- $Q$: Sütunları birbirine dik ve birim uzunluklu (ortogonal) matris  
- $R$: Üst üçgen matris  
Biz sadece $Q$'yu kullanıyoruz.


In [ ]:
def orthogonalize(W):
    """
    W matrisini QR dekompozisyonu ile orthogonalize eder.

    Orthogonalize = sütunlar birbirine dik (ortogonal) hale getir.
    Bu işlem FAVOR+'ın yaklaşım kalitesini artırır.

    Neden? Rastgele vektörler korelasyonlu olabilir → bilgi tekrarı.
    Ortogonal vektörler maksimum bilgi çeşitliliği sağlar.

    Parametreler:
    - W: (n_heads, d_head, m) — orthogonalize edilecek matris
         n_heads başlık, d_head boyut, m feature sayısı

    Döndürür:
    - W_orth: (n_heads, d_head, m) — orthogonalize edilmiş ve ölçeklenmiş W
    """
    d_head = W.size(-2)  # Her başlığın boyutu (örn. 64)

    # W.split(d_head, dim=-1):
    #   W'nin son boyutunu (m) d_head boyutlu parçalara böl
    #   m=256, d_head=64 → 4 chunk: [(n_heads, 64, 64), ...] × 4
    #   Neden d_head büyüklüğünde? d-boyutlu uzayda max d ortogonal vektör var!

    # torch.linalg.qr(W_chunk)[0]:
    #   QR dekompozisyonu uygula: W_chunk = Q * R
    #   [0]: Sadece Q'yu al (R'ye ihtiyaç yok)
    #   Q: Sütunları birbirine dik ve birim uzunluklu matris

    # torch.cat(..., dim=-1):
    #   Tüm orthogonalize edilmiş chunk'ları son boyutta birleştir
    #   (n_heads, 64, 64) × 4 → (n_heads, 64, 256)
    W_orth = torch.cat(
        [torch.linalg.qr(W_chunk)[0]
         for W_chunk in W.split(d_head, dim=-1)],
        dim=-1
    )

    # * d_head**0.5 : √d_head ile ölçekle
    # Neden? QR dekompozisyonu sonrası sütunlar birim uzunluklu (||q||=1).
    # Orijinal Gaussian vektörlerin beklenen normu √d_head'dir.
    # Ölçekleme ile orijinal varyans geri kazanılır → feature map değerleri tutarlı.
    return W_orth * d_head ** 0.5


In [ ]:
# W matrisini orthogonalize et
# W: (8, 64, 256) — daha önce oluşturulan rastgele projeksiyon matrisi
W_orth = orthogonalize(W)
print(f"W boyutu (önce) : {W.shape}")       # (8, 64, 256) — değişmez
print(f"W_orth boyutu   : {W_orth.shape}")  # (8, 64, 256) — aynı boyut
# Boyut aynı kalır! Yalnızca sütunların yönleri değişti (artık ortogonal).


### 3.5 Orthogonalize W ile RMSE — İyileşme Var mı?

In [ ]:
# Orthogonalize W_orth ile FAVOR+ hesabını tekrarla
Qp2 = phi(Q * scale ** 0.5, W_orth, dim_subtract_max=-1)  # Q için ortogonal W
Kp2 = phi(K * scale ** 0.5, W_orth)                        # K için ortogonal W
A2 = Qp2 @ Kp2.transpose(-2, -1)
D2 = A2.sum(dim=-1, keepdim=True)
result2 = A2 / (D2 + 1e-6)

rmse2 = F.mse_loss(result2, expected) ** 0.5
print(f"Rastgele W ile RMSE      : {rmse.item():.6f}")
print(f"Orthogonalize W ile RMSE : {rmse2.item():.6f}")
iyilesme = (rmse - rmse2) / rmse * 100
print(f"İyileşme                 : %{iyilesme.item():.1f}")

# Beklenen: Orthogonalize W daha küçük RMSE verir → daha iyi yaklaşım.
# İyileşme miktarı m değerine ve boyutlara göre değişir.
# m (n_features) artırılırsa her iki yöntemde de RMSE azalır,
# orthogonalizasyonun göreli avantajı ise sabit kalır.


### 3.6 FavorAttention — Tam FAVOR+ Modülü

Tüm FAVOR+ pipeline'ını bir `nn.Module` içinde birleştirelim.

#### Lineer Attention'ın Matematiksel Formülü

Standart:
$$\text{Attention}(Q,K,V) = \underbrace{\text{softmax}\!\left(\frac{QK^T}{\sqrt{d}}\right)}_{L_q \times L_k \text{ matris}} V$$

FAVOR+:
$$\text{Attention}(Q,K,V) \approx \frac{\phi(Q) \cdot (\underbrace{\phi(K)^T V}_{m \times d})}{\phi(Q) \cdot (\underbrace{\Sigma\phi(K)}_{m \times 1})}$$

**Hesaplama sırası kritik:**
- `φ(K)ᵀ @ V` önce: $(m \times L_k) @ (L_k \times d)$ = $(m \times d)$ — küçük!
- `φ(Q) @ (m×d)` sonra: $(L_q \times m) @ (m \times d)$ = $(L_q \times d)$
- Hiçbir zaman $(L_q \times L_k)$ boyutunda matris oluşturulmaz!


In [ ]:
class FavorAttention(nn.Module):
    """
    FAVOR+ Attention — Fast Attention Via Positive Orthogonal Random features.

    Standart O(n²) softmax attention'ı O(n·m) karmaşıklıkta yaklaşık hesaplar.
    Google Brain'in Performer modelinde kullanılan mekanizma.

    Temel fark: Büyük (Lq × Lk) attention matrisi hiç oluşturulmaz.
    Bunun yerine φ(K)ᵀ @ V küçük (m × d) matris üzerinden hesaplanır.

    Parametreler:
    - d_model:    Toplam model embedding boyutu (tüm başlıklar dahil)
    - n_heads:    Multi-head attention başlık sayısı
    - n_features: Random feature sayısı m — büyük m → iyi yaklaşım, yavaş
    """
    def __init__(self, d_model, n_heads, n_features):
        super().__init__()
        self.d_head = d_model // n_heads  # Her başlığın boyutu (örn. 512/8=64)

        # Projeksiyon matrisi W: her başlık için (d_head, n_features)
        W = torch.randn(n_heads, self.d_head, n_features)  # (h, d, m)

        # Orthogonalize: Rastgele vektörleri dik yap → daha iyi yaklaşım
        W = orthogonalize(W)  # (h, d, m) — sütunlar artık ortogonal

        # register_buffer: W'yi model parametresi OLARAK KAYDETME (eğitilmez!)
        # Ama model state'ine dahil et:
        #   - model.state_dict() içinde görünür
        #   - model.to(device) ile GPU'ya taşınır
        #   - model.save_pretrained() ile kaydedilir
        # W rastgele ve sabittir — eğitim sırasında güncellenmez.
        self.register_buffer("W", W)

    def forward(self, Q, K, V):
        """
        FAVOR+ ile yaklaşık attention hesapla.

        Giriş:
        - Q: (B, h, Lq, d_head) — Query tensörü
        - K: (B, h, Lk, d_head) — Key tensörü
        - V: (B, h, Lk, d_head) — Value tensörü

        Döndürür:
        - (B, h, Lq, d_head) — attention çıktısı (standart attention ile aynı boyut)
        """
        # Ölçek faktörü: d_head^(-1/4)
        # Neden -1/4 (standart -1/2 yerine)?
        # φ(√scale·Q) @ φ(√scale·K)ᵀ ≈ exp(Q·Kᵀ·scale)
        # scale = d_head^(-1/2) → √scale = d_head^(-1/4)
        # Her iki tarafı d_head^(-1/4) ile ölçeklemek standart 1/√d attention'a eşdeğer
        scale = self.d_head ** -0.25  # d_head=64 → scale=64^(-0.25)=0.354

        # ── Feature Map Uygula ──
        # Qp: (B, h, Lq, d_head) → (B, h, Lq, m) — Q için feature map
        # dim_subtract_max=-1: Her query kendi max'ına göre normalize edilir
        Qp = phi(Q * scale, self.W, dim_subtract_max=-1)

        # Kp: (B, h, Lk, d_head) → (B, h, Lk, m) — K için feature map
        # dim_subtract_max=(-2,-1): Tüm K'lar için global max (tutarlı normalizasyon)
        Kp = phi(K * scale, self.W)

        # ── Payda: φ(Q) @ Σφ(K) ──
        # Kp.sum(dim=-2): (B, h, Lk, m) → (B, h, m) — tüm K feature'larını topla
        # .unsqueeze(-1): (B, h, m) → (B, h, m, 1) — matmul için boyut ekle
        # Qp @ Kp_sum: (B, h, Lq, m) @ (B, h, m, 1) = (B, h, Lq, 1)
        D = Qp @ Kp.sum(dim=-2).unsqueeze(-1)  # (B, h, Lq, 1) — payda

        # ── Pay: φ(Q) @ (φ(K)ᵀ @ V) ──
        # KRİTİK SIRA: Önce φ(K)ᵀ @ V → (m × d) küçük matris!
        # Kp.transpose(-2, -1): (B, h, Lk, m) → (B, h, m, Lk)
        # @ V: (B, h, m, Lk) @ (B, h, Lk, d_head) = (B, h, m, d_head)
        # Bu matris büyüklüğü: m × d — O(n) değil O(m·d) — n'den bağımsız!
        Kp_T_V = Kp.transpose(-2, -1) @ V  # (B, h, m, d_head)

        # Qp @ Kp_T_V: (B, h, Lq, m) @ (B, h, m, d_head) = (B, h, Lq, d_head)
        # (D + 1e-6): Sıfıra bölme hatası için epsilon
        return (Qp @ Kp_T_V) / (D + 1e-6)  # (B, h, Lq, d_head)


In [ ]:
# FavorAttention modülünü test et
torch.manual_seed(42)

# Test verileri — daha önce kullandığımız boyutlar
Q = torch.randn(batch_size, n_heads, Lq, d_head)   # (32, 8, 200, 64)
K = torch.randn(batch_size, n_heads, Lk, d_head)   # (32, 8, 100, 64)
V = torch.randn(batch_size, n_heads, Lk, d_head)   # (32, 8, 100, 64)

# FavorAttention örneği oluştur
# d_model=512, n_heads=8 → d_head=64
# n_features=256: Projeksiyon boyutu m
favor_attn = FavorAttention(d_model, n_heads, 256)

# Yaklaşık attention hesapla
approx_attn = favor_attn(Q, K, V)

print(f"Giriş Q  : {Q.shape}")           # (32, 8, 200, 64)
print(f"Giriş K  : {K.shape}")           # (32, 8, 100, 64)
print(f"Giriş V  : {V.shape}")           # (32, 8, 100, 64)
print(f"Çıktı    : {approx_attn.shape}") # (32, 8, 200, 64) — standart MHA ile aynı!
print()
print("Büyük attention matrisi (32×8×200×100 = 5.1M eleman) hiç oluşturulmadı ✓")


In [ ]:
import torch.nn.functional as F

# Standart PyTorch attention ile karşılaştır
# F.scaled_dot_product_attention: PyTorch'un optimize edilmiş, CUDA entegrasyonlu
# attention implementasyonu (FlashAttention backend kullanabilir)
attn = F.scaled_dot_product_attention(Q, K, V)

# RMSE: İki yöntemin çıktıları ne kadar yakın?
attn_rmse = F.mse_loss(approx_attn, attn) ** 0.5

print(f"FavorAttention RMSE: {attn_rmse.item():.6f}")
# Tipik değer: 0.005 - 0.05 arası
# Bu değer pratikte kabul edilebilir bir yaklaşım hatası.
# n_features artırılırsa daha küçük RMSE elde edilir (ama hesaplama da artar).
# Gerçek Performer modellerinde n_features=256-1024 arası kullanılır.


---
### 3.7 Sharing Projections in Multi-Head Attention — MQA ve GQA

#### Standart MHA'da K/V Sorunu

Standard Multi-Head Attention'da her başlık kendi **bağımsız** K ve V projeksiyon  
matrislerine sahiptir:

| Bileşen | Boyut | Parametre sayısı (d_model=512, h=8) |
|---------|-------|:-:|
| Q | (B, h, L, d) | 8 × 64 × 64 = 32K |
| K | (B, **h**, L, d) | **8 × 64 × 64 = 32K** |
| V | (B, **h**, L, d) | **8 × 64 × 64 = 32K** |

**KV Cache maliyeti:**  
Her token için tüm katmanlarda K ve V kaydedilir:  
`2 × h × L × d × n_layers × sizeof(dtype)` byte

Bu, büyük modellerde önemli bir darboğaz oluşturur.  
Özellikle **batched inference** (aynı anda birden fazla istek) sırasında bellek sorun olur.

---

### MQA — Multi-Query Attention

**Fikir:** Tüm Q başlıkları **aynı tek K/V** grubunu paylaşır.

```
Standart MHA:   h=8 Q başlığı,  h=8 K grubu,  h=8 V grubu  → 8× K/V
MQA:            h=8 Q başlığı,  1 K grubu,    1 V grubu   → 1× K/V
```

**KV Cache tasarrufu:** 8× daha küçük cache!  
**Parametre tasarrufu:** K ve V matrisleri 8× küçülür  
**Dezavantaj:** Başlıklar arası K/V çeşitliliği sıfır → model kapasitesi azalabilir

**Kullananlar:** PaLM, Falcon, CodeLlama

**PyTorch API:**  
`F.scaled_dot_product_attention(query, key, value, enable_gqa=True)`  
`n_heads / n_groups` oranını otomatik tespit eder.


In [ ]:
# MQA — Multi-Query Attention (n_groups = 1)
batch_size, Lq, Lk, d_head = 32, 100, 90, 64
n_heads = 8     # Query başlık sayısı: 8 bağımsız Q
n_groups = 1    # K/V grup sayısı: TEK grup — tüm Q başlıkları bunu paylaşır

# Query: Her başlık bağımsız → (B, h, Lq, d)
query = torch.randn(batch_size, n_heads, Lq, d_head)   # (32, 8, 100, 64)

# Key: Sadece 1 grup → (B, n_groups, Lk, d)
# Standart MHA'da bu (32, 8, 90, 64) olurdu — şimdi 8× daha küçük!
key   = torch.randn(batch_size, n_groups, Lk, d_head)  # (32, 1, 90, 64)

# Value: Sadece 1 grup → (B, n_groups, Lk, d)
value = torch.randn(batch_size, n_groups, Lk, d_head)  # (32, 1, 90, 64)

# F.scaled_dot_product_attention ile MQA:
# enable_gqa=True: Grouped Query Attention modunu aktive et
# PyTorch içeride şunu yapar:
#   n_heads / n_groups = 8 / 1 = 8 → Her K/V grubu 8 Q başlığına "expand" edilir
#   Ama bu expand bellekte gerçekleşmez (virtual expand) — verimli!
attn_mqa = F.scaled_dot_product_attention(query, key, value, enable_gqa=True)

print("=== MQA (Multi-Query Attention) ===")
print(f"Query  boyutu    : {query.shape}")     # (32, 8, 100, 64) — 8 başlık
print(f"Key    boyutu    : {key.shape}")       # (32, 1, 90, 64)  — 1 grup (8× küçük)
print(f"Value  boyutu    : {value.shape}")     # (32, 1, 90, 64)  — 1 grup (8× küçük)
print(f"Çıktı  boyutu    : {attn_mqa.shape}") # (32, 8, 100, 64) — standart MHA ile aynı!
print(f"KV Cache tasarrufu: {n_heads // n_groups}×")  # 8× daha küçük KV cache


### 3.8 GQA — Grouped Query Attention

**Fikir:** Q başlıkları gruplara ayrılır; her grup kendi K/V'sini paylaşır.  
MHA ve MQA arasında **ayarlanabilir bir denge noktası**.

```
n_heads=8, n_groups=2 durumunda:

Grup 1: Q₁, Q₂, Q₃, Q₄  →  K₁, V₁  (4 Q başlığı 1 K/V paylaşır)
Grup 2: Q₅, Q₆, Q₇, Q₈  →  K₂, V₂  (4 Q başlığı 1 K/V paylaşır)

KV Cache tasarrufu: 4× (MQA'ya göre yarı — ama 2 adet K/V grubu var)
Model kalitesi: MQA'dan iyi, MHA'dan biraz düşük
```

#### Karşılaştırma Tablosu

| Yöntem | K/V grup sayısı | KV Cache | Model Kalitesi |
|--------|:-:|:-:|:-:|
| MHA | h=8 | 1× | En yüksek |
| GQA | n_groups=4 | 2× | Yüksek |
| GQA | n_groups=2 | 4× | İyi |
| MQA | n_groups=1 | 8× | Yeterli |

**GQA'yı kullanan modeller:**  
Llama-2 (n_groups=8), Llama-3 (n_groups=8), Mistral-7B (n_groups=8/32=n_heads/4),  
Gemma (değişken), Falcon-180B (MQA), GPT-4 Turbo (iddia edilen GQA)


In [ ]:
# GQA — Grouped Query Attention (n_groups = 2)
batch_size, Lq, Lk, d_head = 32, 100, 90, 64
n_heads = 8     # 8 bağımsız Q başlığı
n_groups = 2    # 2 K/V grubu — her grup 4 Q başlığına hizmet eder (8/2=4)

# Query: Standart — her başlık bağımsız
query = torch.randn(batch_size, n_heads, Lq, d_head)   # (32, 8, 100, 64)

# Key: 2 grup → (B, n_groups, Lk, d)
# Standart MHA'nın (32, 8, 90, 64) yerine (32, 2, 90, 64) — 4× küçük
key   = torch.randn(batch_size, n_groups, Lk, d_head)  # (32, 2, 90, 64)

# Value: 2 grup
value = torch.randn(batch_size, n_groups, Lk, d_head)  # (32, 2, 90, 64)

# PyTorch GQA hesabı
# enable_gqa=True: n_heads/n_groups = 8/2 = 4 oranı otomatik tespit
# Grup 1 → Q[0:4] başlıkları → key[:, 0], value[:, 0] kullanır
# Grup 2 → Q[4:8] başlıkları → key[:, 1], value[:, 1] kullanır
attn_gqa = F.scaled_dot_product_attention(query, key, value, enable_gqa=True)

print("=== GQA (Grouped Query Attention) ===")
print(f"Query  boyutu    : {query.shape}")      # (32, 8, 100, 64)
print(f"Key    boyutu    : {key.shape}")        # (32, 2, 90, 64) — 4× küçük
print(f"Value  boyutu    : {value.shape}")      # (32, 2, 90, 64)
print(f"Çıktı  boyutu    : {attn_gqa.shape}")  # (32, 8, 100, 64)
print(f"KV Cache tasarrufu: {n_heads // n_groups}×")  # 4×

print()
print("Karşılaştırma özeti:")
print(f"  MHA  (n_groups=8): KV Cache 1×  — maksimum kalite")
print(f"  GQA  (n_groups=2): KV Cache {8//2}×  — iyi denge")
print(f"  MQA  (n_groups=1): KV Cache {8//1}×  — maksimum hız")


---
### 3.9 FlashAttention — IO-Aware Tiling

#### GPU Bellek Hiyerarşisi

FlashAttention, matematik değiştirmeden **donanım seviyesinde** devrim yaratır.

```
GPU Bellek Hiyerarşisi:
┌─────────────────────────────────────────┐
│  SRAM (L1/L2 Cache): ~20-192 KB/SM     │  ← Çok hızlı (~10 ns), ama küçük
│  Tüm GPU SRAM:       ~20 MB            │
├─────────────────────────────────────────┤
│  HBM (GPU RAM — "vRAM"):               │  ← Yavaş (~100 ns = 10× SRAM'dan yavaş)
│  RTX 4090: 24 GB,  A100: 80 GB        │    ama büyük
└─────────────────────────────────────────┘
```

#### Standart Attention'ın IO Darboğazı

```python
# Standart attention — IO açısından verimlisiz:
S = Q @ K.T * scale           # HBM'e yaz: Lq × Lk float
P = softmax(S)                 # HBM'den oku, HBM'e yaz
O = P @ V                      # HBM'den oku
```

**HBM trafiği:** Lq × Lk matrisini 2-3 kez okuyup yaz!
L=1024, d=64 için: 1024² × 3 × 4 byte = **12 MB HBM trafiği** — yavaş!

#### FlashAttention'ın Çözümü

**Tiling (Karolaştırma):**  
Büyük matrisi küçük bloklara böl. Her blok SRAM'a sığar.  
Blok tamamen SRAM'da işlenir, HBM'e asla yazılmaz!

```
Q bloğu (SRAM'a yükle)
  ↓
K/V bloğu 1 (SRAM'a yükle) → kısmi attention → O güncelle → K/V bloğunu at
K/V bloğu 2 (SRAM'a yükle) → kısmi attention → O güncelle → K/V bloğunu at
...
K/V bloğu t (SRAM'a yükle) → kısmi attention → O güncelle → K/V bloğunu at
  ↓
O bloğunu HBM'e yaz (1 kez!)
```

**HBM trafiği:** Sadece Q, K, V okuma (1 kez) + O yazma (1 kez)  
**QKᵀ matrisi hiç HBM'e yazılmaz!**

#### Online Softmax — Tüm Değerleri Görmeden Softmax!

Tiling'in zorluğu: softmax tüm değerleri aynı anda gerektiriyor gibi görünür.  
$$\text{softmax}(s_1, s_2, ..., s_n) = \frac{e^{s_i}}{\sum_j e^{s_j}}$$

Ama online softmax ile blok blok güncellenebilir:

```
Blok 1 geliyor: m₁ = max(s₁...s_k), l₁ = Σexp(s-m₁), O₁ = exp(s-m₁) @ V

Blok 2 geliyor: m₂ = max(m₁, max(s_{k+1}...s_{2k}))
                correction = exp(m₁ - m₂)  ← Eski değerleri yeni max'a uyarla
                l₂ = l₁*correction + Σexp(s-m₂)
                O₂ = O₁*correction + exp(s-m₂) @ V

Son blok sonrası: O_final = O / l  ← Normalize
```

**Matematiksel eşdeğer:** Standart softmax ile tamamen aynı sonuç!


In [ ]:
def flash_attention(Q, K, V, block_size_q, block_size_k):
    """
    FlashAttention algoritmasının basitleştirilmiş Python implementasyonu.

    Matematiksel çıktı standart attention ile TAM OLARAK AYNIDIR.
    Fark: Büyük (Lq × Lk) attention matrisi asla HBM'e yazılmaz.
    Tüm ara hesaplar SRAM boyutundaki bloklarda tutulur.

    !! Bu Python versiyonu eğitim amaçlı. Gerçek FlashAttention bir
       CUDA kernel'i olarak implemente edilir ve GPU'da çok daha hızlıdır. !!

    Kısıtlama: Lq ve Lk'nın sırasıyla block_size_q ve block_size_k'nın
               tam katı olması gerekir.

    Parametreler:
    - Q: (Lq, d) — Query tensörü (tek örnek, başlık yok — basitleştirilmiş)
    - K: (Lk, d) — Key tensörü
    - V: (Lk, d) — Value tensörü
    - block_size_q: Her Q bloğunun satır sayısı (SRAM kapasitesine göre seç)
    - block_size_k: Her K/V bloğunun satır sayısı

    Döndürür:
    - O: (Lq, d) — Attention çıktısı (standart attention ile matematiksel eşdeğer)
    """
    Lq, d = Q.shape   # Query uzunluğu ve boyut
    Lk, _ = K.shape   # Key uzunluğu

    # Çıktı tensörünü sıfır ile başlat (her Q bloğu doldurulacak)
    O = torch.zeros_like(Q)  # (Lq, d)

    # Standart ölçek faktörü: 1/√d
    scale = d ** -0.5

    # ═════════════════════════════════════════════
    # DIŞ DÖNGÜ: Q bloklarını işle
    # Her iterasyon bir Q bloğunu (SRAM'a yüklenir) işler
    # ═════════════════════════════════════════════
    for i_start in range(0, Lq, block_size_q):  # Q bloklarını iterate et
        i_end = min(i_start + block_size_q, Lq)

        # Bu Q bloğunu "SRAM'a yükle"
        Q_block = Q[i_start:i_end]  # (block_size_q, d) — SRAM'da

        # Bu Q bloğu için biriktiricileri başlat (SRAM'da tutulur)
        # O_block: Kısmi attention çıktısı (normalize edilmemiş)
        O_block = torch.zeros(block_size_q, d, device=Q.device)

        # l_block: Softmax paydasının biriktiricisi (Σ exp değerleri)
        # Başlangıçta sıfır — hiç K/V bloğu görmedik
        l_block = torch.zeros(block_size_q, 1, device=Q.device)

        # m_block: Şimdiye kadar görülen maksimum attention skoru
        # -∞ başlangıcı: İlk K/V bloğunda her zaman yeni max = m_ij_new
        m_block = -torch.inf * torch.ones(block_size_q, 1, device=Q.device)

        # ═════════════════════════════════════════════
        # İÇ DÖNGÜ: K/V bloklarını işle
        # Her iterasyon bir K/V bloğunu "SRAM'a yükler", hesaplar ve atar
        # Büyük QKᵀ matrisi ASLA OLUŞTURULMAZ!
        # ═════════════════════════════════════════════
        for j_start in range(0, Lk, block_size_k):  # K/V bloklarını iterate et
            j_end = min(j_start + block_size_k, Lk)

            # K ve V bloklarını "SRAM'a yükle"
            K_block = K[j_start:j_end]  # (block_size_k, d) — SRAM'da
            V_block = V[j_start:j_end]  # (block_size_k, d) — SRAM'da

            # ── Blok Attention Skorları ──
            # S_ij: Bu Q bloğu ile bu K bloğu arasındaki attention skorları
            # Yalnızca (block_size_q × block_size_k) boyutlu küçük matris!
            # Tam QKᵀ matrisi yerine sadece bu karo hesaplanıyor.
            S_ij = Q_block @ K_block.T * scale  # (block_size_q, block_size_k)

            # ── Online Softmax: Maksimum Güncelleme ──
            # Bu bloktan gelen maksimum skor
            m_ij_new, _ = torch.max(S_ij, dim=1, keepdim=True)  # (block_q, 1)

            # Yeni global maksimum: Önceki max ile bu bloğun max'ının büyüğü
            # torch.maximum: Element-wise max (iki tensörü karşılaştırır)
            m_block_new = torch.maximum(m_block, m_ij_new)  # (block_q, 1)

            # ── Online Softmax: Normalize Edilmiş Exponential ──
            # Yeni max'a göre normalize edilmiş exponential değerler
            # exp(S_ij - m_block_new): Taşma önleme — max çıkartılarak küçültme
            P_ij = torch.exp(S_ij - m_block_new)  # (block_q, block_k)

            # ── Düzeltme Faktörü (Correction Factor) ──
            # Eski max'tan yeni max'a geçişte önceki birikim değerleri güncellenmelidir.
            # correction = exp(m_old - m_new)
            # m_old < m_new → m_old - m_new < 0 → correction < 1
            # Yani eski değerler yeni (daha büyük) max'a göre küçültülür.
            # Bu işlem matematiksel olarak softmax normaliz. korur.
            correction_factor = torch.exp(m_block - m_block_new)  # (block_q, 1)

            # ── Payda Güncelleme (l: denominator accumulator) ──
            # l_new = l_old * correction + Σ(yeni exp değerleri)
            # l_old * correction: Eski exp toplamını yeni max'a uyarla
            # torch.sum(P_ij, dim=1): Bu bloktan gelen exp toplamı
            l_block_new = (
                l_block * correction_factor                    # Eski toplamı ölçekle
                + torch.sum(P_ij, dim=1, keepdim=True)        # Yeni exp'leri ekle
            )  # (block_q, 1)

            # ── Çıktı Güncelleme (O: output accumulator) ──
            # O_new = O_old * correction + P_ij @ V_block
            # O_old * correction: Eski çıktıyı yeni max'a uyarla
            # P_ij @ V_block: Bu bloğun weighted value katkısı
            O_block = (
                O_block * correction_factor   # Eski çıktıyı ölçekle
                + (P_ij @ V_block)            # Bu bloğun katkısını ekle
            )  # (block_q, d)

            # Bir sonraki iç döngü iterasyonu için state güncelle
            l_block = l_block_new
            m_block = m_block_new
            # K_block ve V_block artık "SRAM'dan atılır" (Python GC'ye bırakılır)

        # ── Final Normalizasyon ──
        # Tüm K/V blokları işlendi. O_block normalizasyonsuz biriktiricis.
        # l_block: Tüm attention skorlarının Σexp değeri (softmax paydası)
        # Bölme: Normalize edilmiş softmax attention çıktısı
        O[i_start:i_end] = O_block / l_block  # (block_q, d)

        # Bu Q bloğu HBM'e yazıldı. Bir sonraki Q bloğuna geç.

    return O  # (Lq, d) — Standart attention ile matematiksel olarak aynı!


### 3.10 FlashAttention Test Verisi

In [ ]:
# FlashAttention test parametreleri
torch.manual_seed(42)

# Blok boyutları — gerçekte SRAM kapasitesine göre ayarlanır
# Tipik değerler: 64, 128 (GPU'ya göre değişir)
block_size_q = 64   # Her Q bloğundaki satır sayısı
block_size_k = 64   # Her K/V bloğundaki satır sayısı

# Sequence boyutları — blok boyutlarının tam katı olmalı (bu implementasyonda)
Lq = 1280   # Query uzunluğu: 1280 = 64 × 20 (20 Q bloğu)
Lk = 1152   # Key uzunluğu:   1152 = 64 × 18 (18 K bloğu)
d = 512     # Embedding/attention boyutu

# Rastgele test tensörleri
Q = torch.randn(Lq, d)   # (1280, 512)
K = torch.randn(Lk, d)   # (1152, 512)
V = torch.randn(Lk, d)   # (1152, 512)

print(f"Q boyutu: {Q.shape}")
print(f"K boyutu: {K.shape}")
print(f"V boyutu: {V.shape}")
print()
# Standart attention bu boyutlarda ne kadar bellek kullanırdı?
standard_attn_elements = Lq * Lk
standard_attn_bytes = standard_attn_elements * 4  # float32 = 4 byte
print(f"Standart attention matrisi: {Lq} × {Lk} = {standard_attn_elements:,} eleman")
print(f"Standart attention bellek: {standard_attn_bytes / 1024**2:.1f} MB (HBM'e yazılırdı)")
print(f"FlashAttention: Bu matris HBM'e ASLA YAZILMAZ ✓")


In [ ]:
# FlashAttention çıktısını hesapla
R1 = flash_attention(Q, K, V, block_size_q, block_size_k)

# PyTorch'un optimize edilmiş standart attention (referans)
# F.scaled_dot_product_attention: PyTorch 2.0+ ile backend olarak
# gerçek FlashAttention CUDA kernel'ini kullanabilir (otomatik seçim)
R2 = F.scaled_dot_product_attention(Q, K, V)

print(f"FlashAttention çıktısı    : {R1.shape}")
print(f"Standart attention çıktısı: {R2.shape}")


In [ ]:
# İki yöntemin matematiksel eşdeğerliğini doğrula
# MSE ≈ 0 beklenir — aynı matematik, farklı hesaplama düzeni
mse = F.mse_loss(R1, R2)
print(f"MSE (FlashAttention vs. Standart): {mse.item():.2e}")
# float32 hassasiyeti nedeniyle tam 0 olmaz
# Ama 1e-6 ile 1e-9 arasında (floating point hatası) küçük bir değer beklenir
# Bu değer matematiksel eşdeğerliği doğrular ✓


---
##  BÖLÜM 4: Büyük Ölçekte Transformer — MoE ve LoRA

### 4.1 Mixture of Experts (MoE) — Kavramsal Bakış

#### Büyük Model Paradoksu

**Daha büyük model = daha iyi performans** (scaling law)  
Ama sınırsız büyütme çok pahalı inference anlamına gelir.

**Çözüm:** Her token için modelin yalnızca bir **alt kümesini** aktive et!

#### MoE Mimarisi

```
Standart Transformer FFN:
Input → [FFN (tüm parametreler)] → Output

MoE Transformer FFN:
                ┌─ Expert 1 FFN ─┐
                ├─ Expert 2 FFN ─┤
Input → [Gating Network] →  ├─ Expert 3 FFN ─┤ → Weighted Sum → Output
                ├─ Expert 4 FFN ─┤
                ...
                └─ Expert N FFN ─┘
        (Sadece Top-K expert seçilir)
```

**Gating Network:** Her token için hangi K expert'in aktive edileceğine karar verir.  
Genellikle K=2 expert seçilir (Top-2 routing).

**Verimlilik:**
- Toplam parametre: N × expert_count — devasa model
- Forward pass'te aktif parametre: N × K — küçük hesaplama
- Örnek: 64 expert, K=2 → batch başına parametrelerin %3'ü kullanılır!

**MoE kullanan modeller:**  
Mixtral-8x7B (8 expert, 2 aktif), Switch Transformer, GShard, GPT-4 (iddia edilen)

---

### 4.2 LoRA — Low-Rank Adaptation

#### Tam Fine-Tuning'in VRAM Sorunu

7 milyar parametreli bir model fine-tune etmek:

| Bileşen | Parametre başına bellek | 7B parametre için |
|---------|:-:|:-:|
| Ağırlıklar (float16) | 2 byte | ~14 GB |
| Gradyanlar (float32) | 4 byte | ~28 GB |
| Adam optimizer state | 8 byte | ~56 GB |
| **TOPLAM** | | **~100 GB** |

Bu miktarda VRAM tek bir GPU'ya sığmaz!

#### LoRA'nın Matematiksel Temeli

**Gözlem:** Fine-tuning sırasında ağırlık güncellemeleri `ΔW` **düşük ranklidir**.

`ΔW`'nin tekil değer ayrışımında (SVD) yalnızca birkaç büyük tekil değer vardır.  
Geri kalanlar sıfıra yakındır → düşük rank yapı!

**LoRA çözümü:** `ΔW ≈ A × B` (düşük rank ayrışım)

$$W_{\text{finetuned}} = W_{\text{pretrained}} + \Delta W \approx W_{\text{pretrained}} + AB$$

- $W$: $(d_{out} \times d_{in})$ — dondurulmuş orijinal ağırlık
- $A$: $(d_{out} \times r)$ — düşük rankli güncelleme matrisi
- $B$: $(r \times d_{in})$ — düşük rankli güncelleme matrisi
- $r \ll \min(d_{out}, d_{in})$ — genellikle r = 4, 8, 16, 32, 64

**Forward pass:**
$$y = xW^T + \frac{\alpha}{r} \cdot xB^T A^T$$

**Parametre karşılaştırması** (d_out=d_in=768, r=16):
- Orijinal W: 768 × 768 = **590,592 parametre**
- LoRA A+B: (768×16) + (16×768) = **24,576 parametre** → **24× daha az!**

#### Hangi Katmanlara LoRA Uygulanır?

Genellikle attention katmanlarına: Q, V (veya Q, K, V, O hepsi).  
`target_modules=["q_proj", "v_proj"]`

Neden FFN değil? Attention projeksiyonları modelin en kritik bölümlerinden  
birini oluşturur ve düşük rank güncellemeden en çok faydalananlar bunlardır.

#### `lora_alpha` — Ölçekleme Faktörü

`α/r` oranı efektif öğrenme hızını kontrol eder.  
`α = 2r` (örn. r=16 → α=32) yaygın başlangıç değeridir.  
Düşük `α/r`: Ağırlıklı orijinal model (az güncelleme)  
Yüksek `α/r`: Agresif güncelleme (orijinal bilgiyi unutabilir)


In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Temel Model ──
# EleutherAI/gpt-neo-125M: GPT-2 ile aynı mimaride, 125M parametre.
# EleutherAI tarafından açık kaynak olarak yayınlanan model.
# LoRA demostrasyon için ideal boyut (büyük ama GPU'ya sığar).
model_id = "EleutherAI/gpt-neo-125M"

# dtype=torch.float16: Yarı hassasiyet (float16) ile yükle
# float32'ye göre 2× daha az bellek kullanır
# Mixed-precision training ile LoRA ağırlıkları float32'de tutulabilir
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",          # GPU varsa GPU'ya, yoksa CPU'ya
    dtype=torch.float16         # float16 → bellek tasarrufu
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# GPT-Neo'nun ayrı bir PAD token'ı yoktur.
# PAD token olmadan DataLoader padding işlemi yapamaz.
# Çözüm: EOS token'ı (End Of Sequence) PAD olarak kullan.
tokenizer.pad_token = tokenizer.eos_token

# ── LoRA Konfigürasyonu ──
lora_config = LoraConfig(
    r=16,
    # Rank — LoRA matrislerinin boyutu.
    # Düşük r → daha az parametre, daha az ifade gücü.
    # Yüksek r → daha fazla parametre, daha iyi uyum, ama LoRA avantajı azalır.
    # Pratik kural: r=8 çoğu görev için yeterli, r=64 büyük değişiklikler için.

    lora_alpha=32,
    # Ölçekleme faktörü α.
    # Güncelleme: ΔW = (α/r) × A × B
    # α/r = 32/16 = 2 → LoRA güncellemelerini 2× ölçekle.
    # Eğer α=r ise ölçek faktörü 1 olur (nötr).
    # α=2r yaygın başlangıç noktasıdır.

    target_modules=["q_proj", "v_proj"],
    # LoRA'nın uygulanacağı katmanlar (modül isimleri).
    # "q_proj": Query projeksiyon matrisi
    # "v_proj": Value projeksiyon matrisi
    # Bu iki matris attention mekanizmasının en kritik bileşenleri.
    # Daha agresif LoRA için: ["q_proj", "k_proj", "v_proj", "out_proj"] eklenebilir.
    # GPT-Neo'nun katman isimleri için model mimarisine göre ayarla.

    lora_dropout=0.05,
    # LoRA katmanlarına uygulanan dropout oranı.
    # Regularization — overfitting'i önler.
    # 0.05 = %5 dropout (her eğitim adımında nöronların %5'i rastgele kapatılır).
    # Küçük veri setlerinde daha yüksek dropout (0.1-0.2) kullanılabilir.

    bias="none",
    # Bias terimlerini LoRA ile güncelle mi?
    # "none": Bias parametrelerini güncelleme (en yaygın seçim).
    #         Parametre tasarrufunu maksimize eder.
    # "all": Tüm bias'ları güncelle.
    # "lora_only": Yalnızca LoRA katmanlarının bias'larını güncelle.

    task_type="CAUSAL_LM"
    # Görev türü: Causal Language Modeling (GPT stili — soldan sağa üretim).
    # PEFT kütüphanesi bu bilgiyi loss hesabını ve model davranışını ayarlamak için kullanır.
    # Diğer seçenekler: "SEQ_2_SEQ_LM" (T5 gibi), "SEQ_CLS" (sınıflandırma), vb.
)

# ── PEFT Modelini Oluştur ──
# get_peft_model: Üç şey yapar:
# 1. Orijinal model ağırlıklarını DONDURUR (requires_grad=False)
# 2. Belirtilen modüllere LoRA A ve B matrislerini EKLER
# 3. Yalnızca LoRA matrisleri eğitilebilir (requires_grad=True)
peft_model = get_peft_model(model, lora_config)

# Eğitilebilir parametre istatistiklerini göster
# "trainable params": LoRA matrislerinin toplam parametre sayısı
# "all params": Orijinal model + LoRA matrislerinin toplamı
# "trainable%": Eğitilen parametrelerin toplam parametreye oranı
peft_model.print_trainable_parameters()

# Beklenen çıktı (yaklaşık):
# trainable params: 294,912 || all params: 125,543,424 || trainable%: 0.2350
# 125M parametrenin yalnızca %0.24'ü eğitilir → dev tasarruf!


In [ ]:
import torch
from datasets import load_dataset
from transformers import (TrainingArguments, Trainer,
                          DataCollatorForLanguageModeling)

# Bu hücre LoRA ile supervised fine-tuning (SFT) için gerekli
# kütüphaneleri import eder.
#
# ──────────────────────────────────────────────────────────
# TAM LoRA FINE-TUNING PIPELINE (REFERANS KOD):
# ──────────────────────────────────────────────────────────
#
# 1. Veri seti yükleme ve hazırlama:
# dataset = load_dataset("tatsu-lab/alpaca", split="train")
#
# def preprocess(example):
#     return tokenizer(
#         f"### Instruction: {example['instruction']}
"
#         f"### Input: {example['input']}
"
#         f"### Response: {example['output']}",
#         truncation=True, max_length=512
#     )
# tokenized = dataset.map(preprocess, batched=True)
#
# 2. Eğitim argümanları:
# training_args = TrainingArguments(
#     output_dir="./my_lora_model",
#     per_device_train_batch_size=4,   # Küçük batch — LoRA sayesinde mümkün
#     num_train_epochs=3,
#     learning_rate=2e-4,              # LoRA için standart lr (daha yüksek olabilir)
#     fp16=True,                       # Mixed precision training
#     logging_steps=10,
#     save_strategy="epoch",
#     report_to="none"
# )
#
# 3. DataCollator:
# data_collator = DataCollatorForLanguageModeling(
#     tokenizer=tokenizer,
#     mlm=False  # Causal LM için mlm=False (BERT için True)
# )
#
# 4. Trainer:
# trainer = Trainer(
#     model=peft_model,           # LoRA ile sarılmış model
#     args=training_args,
#     train_dataset=tokenized,
#     data_collator=data_collator
# )
# trainer.train()
#
# 5. Yalnızca LoRA ağırlıklarını kaydet (~MB boyutunda):
# peft_model.save_pretrained("./my_lora_model")
# Orijinal model (GB) kaydedilmez — yalnızca LoRA deltası kaydedilir!
#
# 6. Yükleme (inference için):
# from peft import PeftModel
# base_model = AutoModelForCausalLM.from_pretrained(model_id)
# lora_model = PeftModel.from_pretrained(base_model, "./my_lora_model")
# ──────────────────────────────────────────────────────────

print("Import'lar başarıyla yüklendi ✓")
print("peft_model hazır — yukarıdaki referans kodu ile fine-tuning yapılabilir.")


---
## BÖLÜM 5: Hızlı Eğitim — Gradient Accumulation

### Büyük Batch Boyutunun Önemi

**Batch boyutu eğitim kalitesini nasıl etkiler?**

- Küçük batch: Gradient gürültülü → dalgalı kayıp → kararsız eğitim
- Büyük batch: Gradient pürüzsüz → stabil eğitim → genellikle daha iyi model

**Önerilen büyük batch boyutları:**
- LLM pre-training: 1M - 4M token/batch (Llama, GPT-3 kullandı)
- Fine-tuning: 32 - 256 örnek/batch

**Problem:** Bu boyutlar tek GPU'ya sığmaz!

| Model | Batch boyutu | VRAM Gereksinimi |
|-------|:-:|:-:|
| GPT-2 (124M) | 512 | ~50 GB |
| Llama-3 8B | 128 | >300 GB |

### Gradient Accumulation — Küçük Batch'leri Biriktir

**Fikir:** Birden fazla küçük batch'in gradient'larını biriktir.  
Sadece her `accumulation_steps` batch'te bir optimizer adımı at.

```
accumulation_steps = 4, batch_size = 8
→ Efektif batch boyutu = 8 × 4 = 32

Adım 1: Batch[0] → loss/4 → backward() → gradient₁ birikir
Adım 2: Batch[1] → loss/4 → backward() → gradient₁+gradient₂ birikir
Adım 3: Batch[2] → loss/4 → backward() → ... birikir
Adım 4: Batch[3] → loss/4 → backward() → ... birikir
  → optimizer.step()   ← tek optimizer adımı (sanki 32 örnek gördü!)
  → optimizer.zero_grad()
```

**Neden `loss / accumulation_steps` gerekli?**

PyTorch'ta `.backward()` çağrıldığında gradient'lar **toplanır** (accumulate).  
4 backward çağrısı sonrası gradient = gradient₁ + gradient₂ + gradient₃ + gradient₄

Eğer bölmezsek:
- Gerçek büyük batch gradient'ı: `(L₁+L₂+L₃+L₄) / (4N)` = `ortalama loss'un gradienti`
- Bölmeden topladığımızda: `(L₁+L₂+L₃+L₄)` büyüklüğünde gradient
- 4× çok büyük gradient → farklı learning rate etkisi!

Bölünce: Her `loss_i / 4` gradien'tının toplamı = büyük batch gradient'ına eşdeğer.

**Ne DEĞIŞMEZ, ne DEĞİŞİR?**
| | Büyük Batch | Gradient Accumulation |
|--|--|--|
| GPU bellek kullanımı | Büyük (tüm batch) | Küçük (her mini-batch) |
| GPU utilizasyonu | Yüksek | Düşük (batch küçük) |
| Eğitim süresi | - | Biraz daha yavaş |
| Gradient kalitesi | İdeal | Eşdeğer (matematiksel) |
| Final model kalitesi | Referans | Çok benzer |


In [ ]:
# Gradient Accumulation Demo
# Basit lineer model üzerinde kavramı gösteriyoruz.
# Gerçekte: Transformer modelleri için aynı teknik geçerli.

# ── Model ve Optimizer ──
model = torch.nn.Linear(10, 1).to(device)  # Basit lineer model: 10 giriş, 1 çıkış
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()  # Mean Squared Error kayıp fonksiyonu

# Sahte veri: 100 batch, her batch 8 örnek, 10 özellik
data_loader = [(torch.randn(8, 10), torch.randn(8, 1)) for _ in range(100)]

model.train()  # Eğitim moduna geç (dropout, batchnorm aktif)

# ── Accumulation Ayarı ──
accumulation_steps = 4
# Efektif batch boyutu = 8 (gerçek batch) × 4 (accumulation) = 32

# Eğitim başlamadan ÖNCE gradient'ları sıfırla.
# Önemli: Döngü içinde değil, dışında!
optimizer.zero_grad()

for batch_index, (X_batch, y_batch) in enumerate(data_loader):
    # Veriyi uygun device'a taşı
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

    # ── Forward Pass ──
    y_pred = model(X_batch)               # Model tahmini
    loss = criterion(y_pred, y_batch)     # Loss hesapla

    # ── KRİTİK: Loss'u accumulation_steps'e BÖLDÜK ──
    # Neden? Gradient accumulation matematiksel eşdeğerliği için.
    # 4 batch'in loss'unu / 4 ile ölçeklersek:
    #   Toplam gradient = Σ(grad(loss_i/4)) = (1/4) × Σgrad(loss_i)
    #   Bu = grad(Σloss_i / 4) = grad(ortalama loss büyük batch üzerinde)
    # Yani: 32 örneğin ortalamasını sanki tek seferde hesaplamış gibi davranış!
    loss = loss / accumulation_steps

    # ── Backward Pass ──
    # .backward(): Gradient hesapla ve mevcut gradient'lara EKLE (biriktir).
    # optimizer.zero_grad() çağrılmadığı sürece gradient'lar silinmez.
    # Bu biriktirme özelliği gradient accumulation'ı mümkün kılar!
    loss.backward()

    # ── Koşullu Optimizer Adımı ──
    # Her accumulation_steps batch'ten bir kez optimizer.step() çağır.
    # (batch_index + 1) % accumulation_steps == 0:
    #   Batch 3 (index=3): (3+1)%4 = 0 → TRUE → optimizer adımı
    #   Batch 7 (index=7): (7+1)%4 = 0 → TRUE → optimizer adımı
    #   Batch 4 (index=4): (4+1)%4 = 1 → FALSE → sadece biriktir
    if (batch_index + 1) % accumulation_steps == 0:
        # optimizer.step(): Birikmiş gradient'ları kullanarak ağırlıkları güncelle.
        # Bu noktada gradient = 4 mini-batch'in normalleştirilmiş toplamı
        # = 1 büyük batch'in gradient'ına matematiksel eşdeğer
        optimizer.step()

        # optimizer.zero_grad(): Gradient biriktiricileri sıfırla.
        # Bir sonraki accumulation döngüsü için temiz başlangıç.
        optimizer.zero_grad()

print("Eğitim tamamlandı ✓")
print(f"Toplam mini-batch sayısı : {len(data_loader)}")
print(f"Accumulation steps       : {accumulation_steps}")
print(f"Toplam optimizer adımı   : {len(data_loader) // accumulation_steps}")
print(f"Efektif batch boyutu     : {8 * accumulation_steps}")
print(f"Her mini-batch boyutu    : 8 (GPU'ya sığar)")


---
## Bölüm Özeti

### Inference Hızlandırma

| Teknik | Ne Yapar | Hız Artışı | Kalite Etkisi |
|--------|----------|:----------:|:-------------:|
| **KV Caching** | Önceki K/V değerlerini önbellekle | 5-100× | Sıfır |
| **Speculative Dec.** | Draft model tahmin, target doğrular | 2-4× | Sıfır (garanti) |

### Attention Hızlandırma

| Teknik | Bellek Karmaşıklığı | Kalite | Önemli Detay |
|--------|:-:|:-:|---|
| Standart MHA | O(n²) | Tam | Referans |
| **BigBird** | O(n) | Yaklaşık | Uzun diziler için |
| **Reformer (LSH)** | O(n log n) | Yaklaşık | Bucket grupları |
| **Performer (FAVOR+)** | O(n·m) | Yaklaşık | Tüm çiftleri görür |
| **MQA** | O(n²/h) | Yaklaşık | 1 K/V grubu |
| **GQA** | O(n²·g/h) | Yaklaşık | g K/V grubu |
| **FlashAttention** | O(n) HBM | **Tam** | IO-aware |

### Eğitim Hızlandırma

| Teknik | Ne Sağlar | Parametre Azalması |
|--------|-----------|:-:|
| **MoE** | Büyük model, küçük hesaplama | — (routing) |
| **LoRA** | Düşük rankli fine-tuning | ~%99 |
| **Grad. Accum.** | Büyük batch simülasyonu | — |

---

##  Anahtar Kavramlar Sözlüğü

| Kavram | Açıklama |
|--------|----------|
| **KV Cache** | Autoregressive generation'da K/V'leri önceden hesaplamak yerine önbellekte tut |
| **Speculative Decoding** | Küçük draft + büyük target; target'ın kalitesini, draft'ın hızını birleştirir |
| **Sparse Attention** | Her token yalnızca bir alt kümeye dikkat eder (BigBird: yerel+global+rastgele) |
| **LSH** | Benzer vektörleri aynı bucket'a atar; Reformer'da attention gruplamak için kullanılır |
| **FAVOR+** | Gaussian kernel'ı rastgele feature map ile yaklaşık hesaplar; O(n·m) attention |
| **Orthogonalization** | W sütunlarını QR ile birbirine dik yap → daha iyi feature çeşitliliği |
| **MQA** | Tüm Q başlıkları tek K/V paylaşır; büyük KV cache tasarrufu |
| **GQA** | Q başlıkları gruplara ayrılır; MHA ve MQA arasında ayarlanabilir denge |
| **FlashAttention** | Tiling ile QKᵀ matrisini HBM'e yazmadan hesaplar; IO-aware |
| **Online Softmax** | Tüm değerleri görmeden max'ı iteratif güncelle; FlashAttention'ın kalbinde |
| **MoE** | Her token için Top-K expert aktive et; büyük parametre, küçük hesaplama |
| **LoRA** | W_pretrained dondur; ΔW = A×B düşük rankli güncelleme ekle; %99 daha az parametre |
| **lora_alpha** | LoRA güncelleme büyüklüğünü ölçekler: etki = (α/r) × A×B |
| **Gradient Accumulation** | Birden fazla mini-batch gradient'ı biriktir; büyük batch etkisi, küçük bellek |

---

##  Kaynaklar

- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135) (2022)  
- Dao et al., [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691) (2023)  
- Choromanski et al., [Rethinking Attention with Performers](https://arxiv.org/abs/2009.14794) (2020)  
- Kitaev et al., [Reformer: The Efficient Transformer](https://arxiv.org/abs/2001.04451) (2020)  
- Zaheer et al., [Big Bird: Transformers for Longer Sequences](https://arxiv.org/abs/2007.14062) (2020)  
- Ainslie et al., [GQA: Training Generalized Multi-Query Transformer Models](https://arxiv.org/abs/2305.13245) (2023)  
- Hu et al., [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) (2021)  
- Shazeer et al., [Outrageously Large Neural Networks: The Sparsely-Gated MoE](https://arxiv.org/abs/1701.06538) (2017)  
- [Hugging Face PEFT Documentation](https://huggingface.co/docs/peft)
